## setup

In [86]:
from typing import List, Callable, Any, Dict
from pydantic import BaseModel, Field

import os, requests, json, gc, rich
import pandas as pd
import numpy as np

from tqdm import tqdm

import faiss

from openai import OpenAI
from openai.types.chat import ParsedChatCompletion

In [87]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

FCLIP_API_TOKEN = os.getenv("HF_API_TOKEN")
FCLIP_API_ENDPOINT = "https://precove-fclip-back3.hf.space/encode_texts"

BATCH_SIZE = 128
LOAD_FAISS_INDEX = True

In [88]:
os.makedirs("results", exist_ok=True)

In [89]:
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## models

In [90]:
class Product(BaseModel):
    title: str
    description: str

    @classmethod
    def from_row(cls, row: pd.Series) -> "Product":
        return cls(
            title=row["originalTitle"],
            description=row["longDescription"]
        )


class ScoredProduct(Product):
    score: float

In [91]:
class ScoringEval(BaseModel):
    reasoning: str = Field(
        description=(
            "STRICT MAXIMUM 30 WORDS. "
            "Explain exactly why this score was assigned. "
            "Be direct and objective."
        )
    )
    score: int = Field(
        description="Relevance score from 1 to 5.",
        ge=1, le=5
    )


class ScoringEvalResult(BaseModel):
    query: str
    product: ScoredProduct
    response: ScoringEval

In [92]:
class PairwiseEval(BaseModel):
    reasoning: str = Field(
        description=(
            "STRICT MAXIMUM 40 WORDS. "
            "State the deciding factor between the two products, "
            "or the reason for a tie. Do not write an essay."
        )
    )
    best_product_index: int = Field(
        description="Returns 0 if Product 1 is better, 1 if Product 2 is better, or -1 if it is a tie.",
        ge=-1, le=1
    )


class PairwiseEvalResult(BaseModel):
    query: str
    product_0: ScoredProduct
    product_1: ScoredProduct
    response: PairwiseEval

## utils

In [93]:
def load_json(path: str) -> Any:
    with open(path, "r") as f:
        return json.load(f)

In [94]:
def save_json(data: Any, path: str) -> None:
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

In [95]:
def create_embeddings_openai(texts: List[str]) -> List[List[float]]:
    response = client_openai.embeddings.create(
        input=texts,
        model=OPENAI_EMBEDDING_MODEL,
    )
    
    return [item.embedding for item in response.data]

In [96]:
def create_embeddings_fclip(texts: List[str]) -> List[List[float]]:
    headers = {
        "Authorization": f"Bearer {FCLIP_API_TOKEN}",
        "Content-Type": "application/json",
    }
    
    payload = json.dumps({"texts": texts})

    response = requests.request(
        method="POST",
        url=FCLIP_API_ENDPOINT,
        headers=headers,
        data=payload,
    )

    if response.ok:
        return response.json()["embeddings"]

    return None

In [97]:
def create_np_embedding(text: str, embed_func: Callable) -> np.ndarray:
    embedding = embed_func([text])[0]
    np_embedding = np.array(embedding, dtype=np.float32).reshape(1, -1)
    faiss.normalize_L2(np_embedding)

    return np_embedding

In [98]:
def create_np_embeddings(texts: List[str], embed_func: Callable, batch_size: int) -> np.ndarray:
    n, n_success, all_embeddings = 0, 0, []
    loop = tqdm(iterable=range(0, len(texts), batch_size))

    for i in loop:
        n += 1

        try:
            batch = texts[i : i + batch_size]
            batch_embeddings = embed_func(batch)

            if batch_embeddings is not None:
                all_embeddings.extend(batch_embeddings)
                n_success += 1
        
        except Exception as e:
            loop.set_description(str(e))
        
        success_rate = n_success / n
        loop.set_description(f"{success_rate=:.2f}")

    embeddings = np.array(all_embeddings, dtype=np.float32)

    gc.collect()
    del all_embeddings

    return embeddings

In [99]:
def search(
    embedding: np.ndarray,
    index: faiss.IndexFlatIP,
    dataset: List[Product],
    top_k: int
) -> List[ScoredProduct]:
    scores, indices = index.search(embedding, k=top_k)
    results = []

    for score, idx in zip(scores[0], indices[0]):
        product = dataset[idx]

        product = ScoredProduct(
            title=product.title,
            description=product.description,
            score=score
        )
        
        results.append(product)

    return results

In [100]:
def display_search_results(results: List[ScoredProduct]) -> None:
    for rank, product in enumerate(results):
        msg = (
            f"Rank: {rank}\n"
            f"Title: {product.title}\n"
            f"Description: {product.description}\n"
            f"Score: {product.score:.3f}\n"
        )

        rich.print(msg)

## prompts

### system

In [101]:
SYSTEM_PROMPT_SCORING = """You are an expert e-commerce search quality evaluator.
Your task is to judge how relevant a retrieved product is to a user's search query.

# Grading Scale (1-5):
1 - Completely Irrelevant: The product has nothing to do with the query (e.g., query asks for shoes, product is a hat).
2 - Poor Match: Shares a vague category but misses crucial user constraints like gender, specific style, or color.
3 - Fair Match: The product is the right broad type, but misses a secondary but important detail requested by the user.
4 - Good Match: Highly relevant. Fits the core intent and most attributes. Might have a very minor mismatch (e.g., slightly different brand or shade).
5 - Perfect Match: Exact intent. Matches all explicit constraints (type, color, style, brand, usage) flawlessly.

Analyze the query and the product, write a brief reasoning, and then assign the score."""


In [102]:
PAIRWISE_COMPARISON_SYSTEM_PROMPT = """You are an expert e-commerce search quality evaluator.
Your task is to compare two retrieved products against a user's search query and determine which one is the more relevant match.

# Evaluation Criteria:
- Core Intent: Which product better matches the user's primary need?
- Attributes & Constraints: Which product adheres strictly to secondary details (color, brand, style, gender, usage)?
- Specificity: If both are relevant, which one matches the descriptive nuances more precisely?

# Instructions:
1. Briefly analyze how Product 1 matches the query.
2. Briefly analyze how Product 2 matches the query.
3. Compare their strengths and weaknesses relative to the exact query constraints.
4. Assign the final index:
   - 0 : Product 1 is strictly more relevant.
   - 1 : Product 2 is strictly more relevant.
   - -1 : It is a tie (both are equally perfect, equally flawed, or completely irrelevant)."""

### user

In [103]:
def create_user_prompt_scoring(query: str, product: Product) -> str:
    return f"""
User Query: "{query}"

Retrieved Product Title: "{product.title}"
Retrieved Product Description: "{product.description}"

Evaluate the relevance."""

In [104]:
def create_user_prompt_pairwise(
    query: str, 
    product_1: Product, 
    product_2: Product
) -> str:
    return f"""
User Query: "{query}"

--- PRODUCT 1 ---
Title: "{product_1.title}"
Description: "{product_1.description}"

--- PRODUCT 2 ---
Title: "{product_2.title}"
Description: "{product_2.description}"

Compare the two products and output the index of the best match."""

## agents

In [105]:
class OpenAIAgent:
    def __init__(
        self, 
        system_prompt: str,
        output_schema: BaseModel,
        model: str = "gpt-5-mini", 
        temperature: float = 1.
    ):
        self.system_prompt = system_prompt
        self.output_schema = output_schema
        self.model = model
        self.temperature = temperature

        self._response = None
        self._total_prompt_tokens, self._total_completion_tokens = 0, 0
        self._last_prompt_tokens, self._last_completion_tokens = 0, 0

    @property
    def total_prompt_tokens(self) -> int:
        return self._total_prompt_tokens

    @property
    def total_completion_tokens(self) -> int:
        return self._total_completion_tokens

    @property
    def total_tokens(self) -> int:
        return self._total_prompt_tokens + self._total_completion_tokens

    @property
    def last_tokens(self) -> int:
        return self._last_prompt_tokens + self._last_completion_tokens

    @property
    def last_prompt_tokens(self) -> int:
        return self._last_prompt_tokens

    @property
    def last_completion_tokens(self) -> int:
        return self._last_completion_tokens
        
    def generate(self, text: str) -> ParsedChatCompletion:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": text}
        ]
        
        self._response = client_openai.beta.chat.completions.parse(
            model=self.model,
            messages=messages,
            response_format=self.output_schema,
            temperature=self.temperature
        )

        return self._response.choices[0].message.parsed

    def update_tokens(self):
        if self._response is not None:
            self._total_prompt_tokens += self._response.usage.prompt_tokens
            self._total_completion_tokens += self._response.usage.completion_tokens

            self._last_prompt_tokens = self._response.usage.prompt_tokens
            self._last_completion_tokens = self._response.usage.completion_tokens

## dataset

In [106]:
df = pd.read_csv("data/joko_products.csv")
print(df.shape)
df.head()

(17062, 2)


,originalTitle,longDescription
0,Light Blue Wash Fray Waistband Low Waist Strai...,Stay on trend with the light blue wash fray wa...
1,Dark Chocolate Cinched Long Sleeve Denim Jacket,Enhance your aesthetic in this dark chocolate ...
2,Black Diamante Detail Oversized Blazer Dress,We're all about the glam vibes this season and...
3,Petite Black Snatched Sculpt Strappy Maxi Dress,Master the minimalist mood with this black str...
4,Indigo Layered Exposed Pocket Wide Leg Jeans,"Consider these indigo, wide-leg jeans a master..."


In [107]:
products = [
    Product.from_row(row) for _, row in df.iterrows()
]

len(products)

17062

In [108]:
texts_openai = (
    df["originalTitle"].fillna("") + " " +
    df["longDescription"].fillna("")
).str.strip().tolist()

In [109]:
# we only use title since FashionCLIP is limited to 77 tokens
# and has been trained to map pixels to visual keywords that strictly describe clothing

texts_fclip = df["originalTitle"].fillna("").tolist()

## `FAISS` index

In [110]:
if LOAD_FAISS_INDEX:
    index_openai = faiss.read_index("data/index_openai.faiss")

else:
    embeddings_openai = create_np_embeddings(
        texts=texts_openai,
        embed_func=create_embeddings_openai,
        batch_size=BATCH_SIZE
    )
    
    faiss.normalize_L2(embeddings_openai)

    dim_openai = embeddings_openai.shape[1]
    index_openai = faiss.IndexFlatIP(dim_openai)
    index_openai.add(embeddings_openai)

    faiss.write_index(index_openai, "data/index_openai.faiss")

print(f"OpenAI index: {index_openai.ntotal} vectors, dim={index_openai.d}")

OpenAI index: 17062 vectors, dim=1536


In [111]:
if LOAD_FAISS_INDEX:
    index_fclip = faiss.read_index("data/index_fclip.faiss")

else:
    # takes some time to run since FashionCLIP encoder is hosted on Hugging Face CPU Basic space
    embeddings_fclip = create_np_embeddings(
        texts=texts_fclip,
        embed_func=create_embeddings_fclip,
        batch_size=BATCH_SIZE
    )

    faiss.normalize_L2(embeddings_fclip)

    dim_fclip = embeddings_fclip.shape[1]
    index_fclip = faiss.IndexFlatIP(dim_fclip)
    index_fclip.add(embeddings_fclip)

    faiss.write_index(index_fclip, "data/index_fclip.faiss")

print(f"FCLIP index: {index_fclip.ntotal} vectors, dim={index_fclip.d}")

FCLIP index: 17062 vectors, dim=512


## search

In [112]:
TOP_K = 5
query = "nike air max 90 white sneakers"

# "I want to take up yoga, what can I buy?"
# "Ripped jeans with strass"
# "Clothes for summer"
# "Outfit for attending a wedding"
# "Sports clothes for running"

In [113]:
embedding_openai = create_np_embedding(
    text=query, embed_func=create_embeddings_openai
)

results_openai = search(
    embedding=embedding_openai,
    index=index_openai,
    dataset=products,
    top_k=TOP_K
)

display_search_results(results_openai)

Rank: 0
Title: White Ballerina Strap Sneaker Sock
Description: Bring a feminine touch to your basics with the white ballerina strap sneaker sock. Designed in a crisp
white cotton blend with delicate strap detailing, these socks combine comfort with charm. Style with ballet flats 
or sneakers for a polished casual look.
Score: 0.468

Rank: 1
Title: Tall White Ultimate Sweat Cuff Ankle Sweatpants
Description: Elevate your off-duty repertoire with these white, ankle-cuff sweatpants. Crafted from a soft, 
breathable sweat fabric, these sweatpants offer a refined take on relaxed dressing. The cuffed ankle detailing adds
a subtle structure, making them ideal for both lounging and elevated casual edits. Style yours with a tonal ribbed 
top and chunky sneakers for a considered weekend look, or for a monochrome moment, style with one of our corset 
tops for a balanced look. Length approx 84cm/33" (Based on a sample size S) Model wears size S | Tall White 
Ultimate Sweat Cuff Ankle Sweatpants
Score: 0.424

Rank: 2
Title: White Textured Knit Pants
Description: Step up your off-duty day game with these white textured knit pants. Crafted from white textured knit 
material with a relaxed fit and flattering shape. Pair with the matching one shoulder top and chunky sole sandals 
for a put-together daytime look.
Score: 0.421

Rank: 3
Title: Prettylittlethings White Branded Tab Sneaker Socks
Description: Finish off your everyday essentials with these PrettyLittleThing white branded tab sneaker socks. Made
from a white cotton blend material, these ankle socks feature PLT branding for a sporty touch. Pair with sneakers 
and casual fits for off-duty comfort. | Prettylittlethings White Branded Tab Sneaker Socks
Score: 0.421

Rank: 4
Title: White Maxi Pinstripe Tailored Woven Straight Leg Pants
Description: The white maxi pinstripe tailored woven straight leg pants offer a perfect balance of comfort and 
style, designed with a flattering cut and soft fabric for everyday wear. Just add confidence and your favorite 
accessories for a look that we're loving this season.
Score: 0.417

In [114]:
embedding_fclip = create_np_embedding(
    text=query, embed_func=create_embeddings_fclip
)

results_fclip = search(
    embedding=embedding_fclip,
    index=index_fclip,
    dataset=products,
    top_k=TOP_K
)

display_search_results(results_fclip)

Rank: 0
Title: White PLT Branded Sport Socks
Description: The white PLT branded sport socks are a functional yet stylish essential for your everyday or 
activewear wardrobe. Crafted from a soft, breathable cotton-blend fabric with added stretch for comfort and 
flexibility, these socks feature a ribbed cuff and the iconic PLT logo woven in contrast black for a sporty finish.
Whether you're heading to the gym or lounging at home, these socks offer cushioned support and a snug fit. Style 
them peeking out from chunky Sneakers, paired with biker shorts and an oversized Sweatshirt for an effortlessly 
cool, athleisure-inspired look.
Score: 0.607

Rank: 1
Title: White Mid Waist Straight Leg Jeans
Description: Keep it classic with the white mid waist straight leg jeans. Crafted from clean white denim fabric 
with a mid-rise waist and straight leg fit, these jeans are a wardrobe essential. Style with a the matching top and
heels for an effortlessly chic finish.
Score: 0.602

Rank: 2
Title: White Woven Fitted Shirt
Description: Add this white woven fitted shirt to your new season wardrobe for a day to day staple we're loving. 
Brought to you in a crisp white shade, with a luxe woven material and flattering fitted design, this shirt is a 
must have. Team with shorts, a pair of tights, court heels and a shoulder bag for a look that won't go unnoticed.
Score: 0.600

Rank: 3
Title: White High Waist Straight Leg Jeans
Description: Opt for a flawless finish in these white high waist straight leg jeans. Tailored from clean white 
denim, these jeans have a high waisted fit with straight legs. Pair it with a striped sweater simple flat sandals 
and gold earrings.
Score: 0.596

Rank: 4
Title: White Long Sleeve Top
Description: Refresh your wardrobe essentials with this white long sleeve top. Made from a crisp white fabric in a 
timeless long sleeve style. Level up your look by styling it with the matching pants, textural ballet flats and 
understated gold jewelry.
Score: 0.594

## evaluation

In [115]:
agent_scoring = OpenAIAgent(
    system_prompt=SYSTEM_PROMPT_SCORING,
    output_schema=ScoringEval,
    model="gpt-5-mini",
)

agent_pairwise = OpenAIAgent(
    system_prompt=PAIRWISE_COMPARISON_SYSTEM_PROMPT,
    output_schema=PairwiseEval,
    model="gpt-5-mini",
)     

### single example

In [116]:
idx = 0
product_openai = results_openai[idx]
product_fclip = results_fclip[idx]

In [117]:
user_prompt_openai = create_user_prompt_scoring(
    query=query, product=product_openai
)

rich.print(user_prompt_openai)
print("-" * 100)

response_openai = agent_scoring.generate(text=user_prompt_openai)
agent_scoring.update_tokens()

print(f"Score: {response_openai.score}")
rich.print(response_openai.reasoning)
print(f"Tokens: {agent_scoring.last_tokens}")

User Query: "nike air max 90 white sneakers"

Retrieved Product Title: "White Ballerina Strap Sneaker Sock"
Retrieved Product Description: "Bring a feminine touch to your basics with the white ballerina strap sneaker sock. 
Designed in a crisp white cotton blend with delicate strap detailing, these socks combine comfort with charm. Style
with ballet flats or sneakers for a polished casual look."

Evaluate the relevance.

----------------------------------------------------------------------------------------------------
Score: 1


Item is a white ballerina-style sock, not Nike Air Max 90 sneakers—different category and brand; fails to meet 
user’s requested shoes.

Tokens: 674


In [118]:
user_prompt_fclip = create_user_prompt_scoring(
    query=query, product=product_fclip
)

rich.print(user_prompt_fclip)
print("-" * 100)

response_fclip = agent_scoring.generate(text=user_prompt_fclip)
agent_scoring.update_tokens()

print(f"Score: {response_fclip.score}")
rich.print(response_fclip.reasoning)
print(f"Tokens: {agent_scoring.last_tokens}")

User Query: "nike air max 90 white sneakers"

Retrieved Product Title: "White PLT Branded Sport Socks"
Retrieved Product Description: "The white PLT branded sport socks are a functional yet stylish essential for your 
everyday or activewear wardrobe. Crafted from a soft, breathable cotton-blend fabric with added stretch for comfort
and flexibility, these socks feature a ribbed cuff and the iconic PLT logo woven in contrast black for a sporty 
finish. Whether you're heading to the gym or lounging at home, these socks offer cushioned support and a snug fit. 
Style them peeking out from chunky Sneakers, paired with biker shorts and an oversized Sweatshirt for an 
effortlessly cool, athleisure-inspired look."

Evaluate the relevance.

----------------------------------------------------------------------------------------------------
Score: 1


Socks, not shoes; not Nike Air Max 90 or sneakers. Different product type and brand — completely irrelevant.

Tokens: 668


In [119]:
user_prompt_pairwise = create_user_prompt_pairwise(
    query=query, 
    product_1=product_openai, 
    product_2=product_fclip
)

rich.print(user_prompt_fclip)
print("-" * 100)

response_pairwise = agent_pairwise.generate(text=user_prompt_pairwise)
agent_pairwise.update_tokens()

print(f"Best Product: {response_pairwise.best_product_index}")
rich.print(response_pairwise.reasoning)
print(f"Tokens: {agent_pairwise.last_tokens}")

User Query: "nike air max 90 white sneakers"

Retrieved Product Title: "White PLT Branded Sport Socks"
Retrieved Product Description: "The white PLT branded sport socks are a functional yet stylish essential for your 
everyday or activewear wardrobe. Crafted from a soft, breathable cotton-blend fabric with added stretch for comfort
and flexibility, these socks feature a ribbed cuff and the iconic PLT logo woven in contrast black for a sporty 
finish. Whether you're heading to the gym or lounging at home, these socks offer cushioned support and a snug fit. 
Style them peeking out from chunky Sneakers, paired with biker shorts and an oversized Sweatshirt for an 
effortlessly cool, athleisure-inspired look."

Evaluate the relevance.

----------------------------------------------------------------------------------------------------
Best Product: -1


Tie — neither product is Nike Air Max 90 white sneakers. Both are white socks, not shoes, so both are equally 
irrelevant to the query.

Tokens: 914


## multiple examples

In [120]:
def run_pairwise_evaluation(
    queries: List[str],
    top_k: int,
    save_path: str | None = None,
) -> List[PairwiseEvalResult]:
    results: List[PairwiseEvalResult] = []
    results_json: List[Dict[str, Any]] = []
    loop = tqdm(iterable=queries)

    for query in loop:
        embeddings_openai = create_np_embedding(query, create_embeddings_openai)
        embeddings_fclip  = create_np_embedding(query, create_embeddings_fclip)

        products_openai = search(embeddings_openai, index_openai, products, top_k=top_k)
        products_fclip  = search(embeddings_fclip,  index_fclip,  products, top_k=top_k)

        for idx in range(top_k):
            product_openai = products_openai[idx]
            product_fclip  = products_fclip[idx]

            prompt = create_user_prompt_pairwise(
                query=query,
                product_1=product_openai,
                product_2=product_fclip,
            )

            response: PairwiseEval = agent_pairwise.generate(text=prompt)
            agent_pairwise.update_tokens()

            eval_result = PairwiseEvalResult(
                query=query,
                product_0=product_openai,
                product_1=product_fclip,
                response=response,
            )
            
            results.append(eval_result)

        if save_path is not None:
            json_data = eval_result.model_dump()
            results_json.append(json_data)
            save_json(data=results_json, path=save_path)

    return results

In [121]:
dataset_simple = load_json("queries/simple.json")
dataset_thematic = load_json("queries/thematic.json")
dataset_specific = load_json("queries/specific.json")    

In [122]:
for dataset in [dataset_simple, dataset_thematic, dataset_specific]:
    print(dataset["category_name"], ":", dataset["description"], f"(n={len(dataset['queries'])})")
    print(dataset["queries"][0])
    print()

simple : Short queries focusing on basic clothing types and generic colors or styles. (n=30)
white cotton t-shirt

thematic : Queries describing a specific style, aesthetic, event, or weather context. (n=30)
trendy outfit for spring

specific : Queries focusing on specific exact models, technical materials, strict construction constraints, and occasional specific brands. (n=30)
levis 501 original fit men's jeans



In [ ]:
results_simple = run_pairwise_evaluation(
    queries=dataset_simple["queries"],
    top_k=1,
    save_path="results/pairwise_simple.json"
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
agent_pairwise.total_prompt_tokens, agent_pairwise.total_completion_tokens